In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("job_lists_export.csv" , delimiter= ";")  # Read in the job listings CSV file
df.head(1)

,id,title,user_id,location,date_posted,description,skills,min_salary,max_salary,salary_negotiable,payment_period,payment_currency,hiring_multiple_candidates
0,1,"Application Developer - ERP, Commercial CPS",12287,"Atlanta, GA, US",2025-04-15,Cargill’s size scale allows us make positive i...,"acceptance testing,application,application sup...",71538,91833,1,monthly,USD,0


In [3]:
df["full_text"] = df["title"] + ",job_description:" + df["description"]
df.head(1)

,id,title,user_id,location,date_posted,description,skills,min_salary,max_salary,salary_negotiable,payment_period,payment_currency,hiring_multiple_candidates,full_text
0,1,"Application Developer - ERP, Commercial CPS",12287,"Atlanta, GA, US",2025-04-15,Cargill’s size scale allows us make positive i...,"acceptance testing,application,application sup...",71538,91833,1,monthly,USD,0,"Application Developer - ERP, Commercial CPS,jo..."


In [4]:
from sentence_transformers import SentenceTransformer , util
model = SentenceTransformer('./job_embeddings_model_evaluated2')


d:\anaconda\envs\REC\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
job_embeddings2 = model.encode(df['full_text'].tolist(), show_progress_bar=True)

Batches: 100%|██████████| 590/590 [04:43<00:00,  2.08it/s]


In [6]:
import pickle

# Save embeddings
with open('job_embeddings.pkl', 'wb') as f:
    pickle.dump(job_embeddings2, f)
print("Embeddings saved to job_embeddings.pkl")

Embeddings saved to job_embeddings.pkl


In [7]:
with open('job_embeddings.pkl', 'rb') as f:
    loaded_embeddings = pickle.load(f)
print("Embeddings loaded from job_embeddings.pkl")

Embeddings loaded from job_embeddings.pkl


In [20]:
import pandas as pd

data = pd.read_csv("E:/vs codes/REC/data/job_lists.csv" , delimiter=';')  # Specify delimiter=';' like you did with your other CSV file
data.head(1)

,id,title,user_id,location,date_posted,description,skills,min_salary,max_salary,salary_negotiable,payment_period,payment_currency,hiring_multiple_candidates,created_at,updated_at
0,1,Senior Frontend Developer,4,"Menlo Park, California",2025-06-11,<h2>About This Role</h2>\n<p>We are looking fo...,"[""TypeScript"",""HTML"",""CSS"",""Angular""]",80000,120000,1,yearly,USD,1,2025-07-01 20:02:51,2025-07-02 11:36:17


In [3]:
data['skills_text'] = data['skills'].apply(lambda x: ", ".join(x) if isinstance(x, list) else str(x))
data.head(1)

,id,title,user_id,location,date_posted,description,skills,min_salary,max_salary,salary_negotiable,payment_period,payment_currency,hiring_multiple_candidates,created_at,updated_at,skills_text
0,1,Senior Frontend Developer,4,"Menlo Park, California",2025-06-11,<h2>About This Role</h2>\n<p>We are looking fo...,"[""TypeScript"",""HTML"",""CSS"",""Angular""]",80000,120000,1,yearly,USD,1,2025-07-01 20:02:51,2025-07-02 11:36:17,"[""TypeScript"",""HTML"",""CSS"",""Angular""]"


In [21]:
import ast

# Function to convert string representation of list to actual string
def extract_skills_as_string(skills_value):
    try:
        # If it's already a list, join it
        if isinstance(skills_value, list):
            return ", ".join(skills_value)
        
        # If it's a string representation of a list, convert then join
        elif isinstance(skills_value, str):
            # Try to evaluate as a Python list
            try:
                skills_list = ast.literal_eval(skills_value)
                if isinstance(skills_list, list):
                    return ", ".join(skills_list)
            except:
                pass
            
            # If it's not a valid list representation or conversion fails
            # Strip brackets if it looks like a list
            if skills_value.startswith('[') and skills_value.endswith(']'):
                skills_value = skills_value[1:-1]
                # Split by comma and clean up quotes
                skills_list = [s.strip().strip('"\'') for s in skills_value.split(',')]
                return ", ".join(skills_list)
        
        # Default case
        return str(skills_value)
    except:
        return str(skills_value)

# Apply the function to create a clean skills text column
data['skills_text'] = data['skills'].apply(extract_skills_as_string)

# Show example
print("Original:", data['skills'].iloc[0])
print("Converted:", data['skills_text'].iloc[0])

Original: ["TypeScript","HTML","CSS","Angular"]
Converted: TypeScript, HTML, CSS, Angular


<unknown>:1: SyntaxWarning: invalid escape sequence '\/'
<unknown>:1: SyntaxWarning: invalid escape sequence '\/'


In [24]:
data.head(1)

,id,title,user_id,location,date_posted,description,skills,min_salary,max_salary,salary_negotiable,payment_period,payment_currency,hiring_multiple_candidates,created_at,updated_at,skills_text
0,1,Senior Frontend Developer,4,"Menlo Park, California",2025-06-11,<h2>About This Role</h2>\n<p>We are looking fo...,"[""TypeScript"",""HTML"",""CSS"",""Angular""]",80000,120000,1,yearly,USD,1,2025-07-01 20:02:51,2025-07-02 11:36:17,"TypeScript, HTML, CSS, Angular"


In [25]:
data = data[["id" , "title" , "description" , "skills_text"]]
data.head(1)

,id,title,description,skills_text
0,1,Senior Frontend Developer,<h2>About This Role</h2>\n<p>We are looking fo...,"TypeScript, HTML, CSS, Angular"


In [26]:
data["full_text"] = data["title"] + ",job_description:" + data["description"] + ",skills:" + data["skills_text"]
data.head(1)

C:\Users\basil\AppData\Local\Temp\ipykernel_18560\1224316820.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["full_text"] = data["title"] + ",job_description:" + data["description"] + ",skills:" + data["skills_text"]


,id,title,description,skills_text,full_text
0,1,Senior Frontend Developer,<h2>About This Role</h2>\n<p>We are looking fo...,"TypeScript, HTML, CSS, Angular","Senior Frontend Developer,job_description:<h2>..."


In [27]:
data.loc[5 , "description"] 

'<h2>About This Role</h2>\n<p>We are seeking a Technical Project Manager to lead our development projects. You will be responsible for planning, executing, and closing projects according to strict deadlines and within budget.</p>\n\n<h3>Key Responsibilities:</h3>\n<ul>\n  <li>Plan and manage the full project lifecycle</li>\n  <li>Define project scope, goals, and deliverables</li>\n  <li>Coordinate internal resources and third parties/vendors</li>\n  <li>Manage project budget and resource allocation</li>\n  <li>Communicate project status to stakeholders</li>\n</ul>\n\n<h3>Requirements:</h3>\n<ul>\n  <li>4+ years of experience in technical project management</li>\n  <li>PMP or other project management certification</li>\n  <li>Understanding of software development processes</li>\n  <li>Strong leadership and communication skills</li>\n  <li>Experience with project management tools</li>\n</ul>'

In [29]:
# Install if needed
# !pip install beautifulsoup4

from bs4 import BeautifulSoup

# Function to clean HTML from text
def clean_html_text(html_text):
    soup = BeautifulSoup(html_text, 'html.parser')
    
    # Extract text while preserving some structure with line breaks
    headings = soup.find_all(['h1', 'h2', 'h3', 'h4', 'h5', 'h6'])
    for heading in headings:
        # Add line breaks before and after headings
        heading.insert_before('\n\n')
        heading.append('\n')
    
    # Add line breaks after paragraphs and list items
    for tag in soup.find_all(['p', 'li']):
        tag.append('\n')
    
    # Replace bullet points
    for li in soup.find_all('li'):
        li.insert_before('• ')
    
    # Get text and clean up extra whitespace
    text = soup.get_text()
    text = '\n'.join([line.strip() for line in text.split('\n') if line.strip()])
    
    return text

# Add this to your dataframe
data['clean_description'] = data['description'].apply(clean_html_text)

# Display an example
print(clean_html_text(data.loc[5, "description"]))

# Now use this in your full_text
data["full_text"] = data["title"] + ", " + data["clean_description"] + ", Skills: " + data["skills_text"]

About This Role
We are seeking a Technical Project Manager to lead our development projects. You will be responsible for planning, executing, and closing projects according to strict deadlines and within budget.
Key Responsibilities:
• Plan and manage the full project lifecycle
• Define project scope, goals, and deliverables
• Coordinate internal resources and third parties/vendors
• Manage project budget and resource allocation
• Communicate project status to stakeholders
Requirements:
• 4+ years of experience in technical project management
• PMP or other project management certification
• Understanding of software development processes
• Strong leadership and communication skills
• Experience with project management tools


C:\Users\basil\AppData\Local\Temp\ipykernel_18560\596816824.py:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['clean_description'] = data['description'].apply(clean_html_text)
C:\Users\basil\AppData\Local\Temp\ipykernel_18560\596816824.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["full_text"] = data["title"] + ", " + data["clean_description"] + ", Skills: " + data["skills_text"]


In [31]:
data.loc[0, 'full_text']

'Senior Frontend Developer, About This Role\nWe are looking for a Senior Frontend Developer to join our dynamic team. You will be responsible for building and maintaining user interfaces for our web applications.\nKey Responsibilities:\n• Develop new user-facing features using React.js and modern frontend technologies\n• Build reusable components and libraries for future use\n• Translate designs and wireframes into high-quality code\n• Optimize applications for maximum speed and scalability\n• Collaborate with back-end developers and web designers\nRequirements:\n• 5+ years of experience in frontend development\n• Proficiency with JavaScript, HTML5, CSS3\n• Experience with React.js or similar frontend frameworks\n• Understanding of responsive design principles\n• Knowledge of modern frontend build pipelines and tools., Skills: TypeScript, HTML, CSS, Angular'

In [32]:
data = data[["id" , "full_text"]]
data.head(1)

,id,full_text
0,1,"Senior Frontend Developer, About This Role\nWe..."


In [33]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('./job_embeddings_model_evaluated2')

In [34]:
job_embeddings2 = model.encode(data['full_text'].tolist(), show_progress_bar=True)

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.70it/s]


In [36]:
import pickle

# Save embeddings
with open('job_embeddings2.pkl', 'wb') as f:
    pickle.dump(job_embeddings2, f)
print("Embeddings saved to job_embeddings1.pkl")

Embeddings saved to job_embeddings1.pkl
